In [ ]:
import numpy as np
import sys
import os

sys.path.append(os.path.abspath("../"))

from src.core import generate_space, generate_random_basis
from src.geometries import generate_hamming_ball
from src.operators import compute_sumset, find_maximum_subspace_dimension
from src.covers import generate_covering, complement

Initialize Universe $\mathbb{F}_2^4

In [5]:
n = 4
p = 2
universe = generate_space(n, p)

print(f"Total elements in universe: {len(universe)}")

Total elements in universe: 16


%% Run an Experiment: Can 4 Symmetric Hamming Balls of Radius 1 completely cover F_2^4?

In [6]:
np.random.seed(42)

# Generates 4 distinct random bases and centers
bases = [generate_random_basis(n, p) for _ in range(4)]
centers = [universe[np.random.choice(len(universe))] for _ in range(4)]
radius = 1

Compute individual balls

In [ ]:
covered = generate_covering(centers, radius, bases=bases, p=p, universe=universe)
S = complement(universe, covered)

print(f"Elements covered by 4 balls: {len(covered)} / {2**n}")
print(f"Size of leftover structured set S: {len(S)}")

Elements covered by 4 balls: 10 / 16
Size of leftover structured set S: 6


%% Evaluate the additive structure of S if it isn't empty

In [8]:
if len(S) > 0:
    S_plus_s = compute_sumset(S, p)
    max_dim = find_maximum_subspace_dimension(S_plus_s, p)
    print(f"Size of S+S: {len(S_plus_s)}")
    print(f"Dimension of largest subspace inside S+S: {max_dim} (Codimension: {n - max_dim})")
else:
    print("Perfect covering achieved! This specific basis configuration serves as a 4-ball counterexample.")

Size of S+S: 13
Dimension of largest subspace inside S+S: 4 (Codimension: 0)


Track 1: Testing 4+ Hamming Balls (Fixed: p=2, r=1)

In [ ]:
n=6
p=2
universe = generate_space(n,p)

print(f"Total elements in the universe: {len(universe)}")




Track 2: Testing Asymmtric Radii (Fixed K=3, p=2)

Track 3: Testing Higher Characteristics $\mathbb{F}_p^n$ (Fixed K, equal radii)

## Experiment: K=3..6 balls, n=5 and n=6, radius=1
This cell runs 1000 randomized trials per (n,K) with independent random invertible bases and random centers. It records the size of the leftover set S and the dimension of the largest subspace inside S+S. Look for `S_size==0` (zero-hole) and for sudden codimension drops.

In [ ]:
import time
import json
np.random.seed(0)

def run_experiments(ns=(5,6), Ks=(3,4,5,6), trials=1000, radius=1, p=2, show_progress=True):
    results = {}
    zero_hole_records = []
    for n in ns:
        universe = generate_space(n, p)
        universe_tuples = [tuple(row) for row in universe]
        results[n] = {}
        for K in Ks:
            if show_progress: print(f'Running n={n}, K={K}, trials={trials}')
            res_list = []
            start = time.time()
            for t in range(trials):
                # generate K independent random invertible bases and random centers
                bases = [generate_random_basis(n, p) for _ in range(K)]
                centers = [universe[np.random.choice(len(universe))] for _ in range(K)]
                try:
                    covered = generate_covering(centers, radius, bases=bases, p=p, universe=universe)
                except Exception:
                    # if generation fails for some reason, skip this trial
                    continue

                S = complement(universe, covered)
                S_size = len(S)
                if S_size == 0:
                    max_dim = None
                    zero_hole_records.append({'n': n, 'K': K, 'trial': t, 'centers': [list(map(int,c)) for c in centers]})
                else:
                    S_plus_s = compute_sumset(S, p)
                    max_dim = find_maximum_subspace_dimension(S_plus_s, p)
                res_list.append({'S_size': S_size, 'max_subspace_dim': max_dim})
                # optional progress print
                if show_progress and (t+1) % 200 == 0:
                    elapsed = time.time() - start
                    print(f'  trial {t+1}/{trials} elapsed={elapsed:.1f}s')
            results[n][K] = res_list
    return results, zero_hole_records

# Run the experiments (this will take some time) — adjust `trials` if you wish to run fewer samples while testing
exp_results, zero_holes = run_experiments(ns=(5,6), Ks=(3,4,5,6), trials=1000, radius=1, p=2, show_progress=True)

# Summarize and save
summary = {}
for n, Ks_res in exp_results.items():
    summary[n] = {}
    for K, records in Ks_res.items():
        S_sizes = [r['S_size'] for r in records]
        dims = [r['max_subspace_dim'] for r in records if r['max_subspace_dim'] is not None]
        summary[n][K] = {
            'trials_run': len(S_sizes),
            'S_size_mean': float(np.mean(S_sizes)) if len(S_sizes)>0 else None,
            'S_size_median': float(np.median(S_sizes)) if len(S_sizes)>0 else None,
            'S_size_zero_count': int(sum(1 for s in S_sizes if s==0)),
            'max_subspace_dim_mean': float(np.mean(dims)) if len(dims)>0 else None,
            'max_subspace_dim_median': float(np.median(dims)) if len(dims)>0 else None
        }
# print concise summary
print('--- Experiment Summary ---')
for n, Ks_res in summary.items():
    print(f'n={n}')
    for K, stats in Ks_res.items():
        print(f"  K={K}: trials={stats['trials_run']}, zeros={stats['S_size_zero_count']}, S_mean={stats['S_size_mean']}, subspace_dim_mean={stats['max_subspace_dim_mean']}")

# save results to disk
import pathlib
out_path = pathlib.Path('notebooks') / 'k_balls_experiment_results.npz'
np.savez_compressed(out_path, exp_results=exp_results, zero_holes=zero_holes, summary=summary)
print(f'Results saved to {out_path}')